# M0 — Identity synthetic-data control

This notebook creates the no-noise control for the synthetic-data benchmark. Each formal Wolof source is copied unchanged into the input column. M0 is not expected to normalize informal text; it measures what a model learns when its synthetic training data teaches only copying.

The output uses the same schema as M1 and the later alignment/LLM/YouTube-aware methods. The manifest records input and output checksums.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    M0_IDENTITY_MANIFEST_PATH, M0_IDENTITY_PATH, TRAIN_PARQUET_PATH
)
from src.generation.benchmark_baselines import (
    METHOD_M0, build_identity_pairs, load_formal_sources,
    write_dataset_and_manifest,
)

## Configuration
During development, `OVERWRITE=True` makes Run All reproducible. Set it to `False` after the benchmark artifacts are frozen.

In [2]:
INPUT_PATH = TRAIN_PARQUET_PATH
OUTPUT_PATH = M0_IDENTITY_PATH
MANIFEST_PATH = M0_IDENTITY_MANIFEST_PATH
OVERWRITE = True

print(f'Input:    {INPUT_PATH}')
print(f'Output:   {OUTPUT_PATH}')
print(f'Manifest: {MANIFEST_PATH}')

Input:    C:\Users\thioy\Desktop\School\PFE\data\translation data\train-00000-of-00001.parquet
Output:   C:\Users\thioy\Desktop\School\PFE\data\synthetic\m0_identity_pairs.csv
Manifest: C:\Users\thioy\Desktop\School\PFE\data\synthetic\m0_identity_manifest.json


## Load the common formal source pool

In [3]:
sources = load_formal_sources(INPUT_PATH)
print(f'Formal source rows: {len(sources):,}')
print(f'Unique source IDs:  {sources.source_id.nunique():,}')
sources.head()

Formal source rows: 17,777
Unique source IDs:  17,777


,source_id,source_row,formal_wolof
0,parallel:0,0,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...
1,parallel:1,1,"""Amuñu woon benn jot ngir rawal sunu bopp."
2,parallel:2,2,"Ekost sa dëkk la te bëggna nu nga toog fi."""
3,parallel:3,3,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...
4,parallel:4,4,ñaata at nga am


## Generate and validate M0

In [4]:
m0 = build_identity_pairs(sources)

assert len(m0) == len(sources)
assert m0.source_id.is_unique
assert m0.formal_wolof.notna().all() and m0.informal_wolof.notna().all()
assert m0.formal_wolof.equals(m0.informal_wolof)
assert not m0.changed.any()
assert (m0.edit_count == 0).all()

m0.head(10)

,source_id,source_row,source_corpus,method,seed,informal_wolof,formal_wolof,changed,edit_count,error_types,applied_rules
0,parallel:0,0,french_wolof_parallel,m0_identity,2026,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,False,0,[],[]
1,parallel:1,1,french_wolof_parallel,m0_identity,2026,"""Amuñu woon benn jot ngir rawal sunu bopp.","""Amuñu woon benn jot ngir rawal sunu bopp.",False,0,[],[]
2,parallel:2,2,french_wolof_parallel,m0_identity,2026,"Ekost sa dëkk la te bëggna nu nga toog fi.""","Ekost sa dëkk la te bëggna nu nga toog fi.""",False,0,[],[]
3,parallel:3,3,french_wolof_parallel,m0_identity,2026,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,False,0,[],[]
4,parallel:4,4,french_wolof_parallel,m0_identity,2026,ñaata at nga am,ñaata at nga am,False,0,[],[]
5,parallel:5,5,french_wolof_parallel,m0_identity,2026,"Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan Nja...","Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan Nja...",False,0,[],[]
6,parallel:6,6,french_wolof_parallel,m0_identity,2026,Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm...,Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm...,False,0,[],[]
7,parallel:7,7,french_wolof_parallel,m0_identity,2026,"Zambie réew la mu bokk ci ONU, Union Africaine...","Zambie réew la mu bokk ci ONU, Union Africaine...",False,0,[],[]
8,parallel:8,8,french_wolof_parallel,m0_identity,2026,tooy,tooy,False,0,[],[]
9,parallel:9,9,french_wolof_parallel,m0_identity,2026,"Ay teemeeri nit faatu ca donu bu Nias, ci tefe...","Ay teemeeri nit faatu ca donu bu Nias, ci tefe...",False,0,[],[]


In [5]:
print(m0[['changed', 'edit_count']].describe(include='all'))
print(f'Exact duplicate input strings: {m0.informal_wolof.duplicated().sum():,}')

       changed  edit_count
count    17777     17777.0
unique       1         NaN
top      False         NaN
freq     17777         NaN
mean       NaN         0.0
std        NaN         0.0
min        NaN         0.0
25%        NaN         0.0
50%        NaN         0.0
75%        NaN         0.0
max        NaN         0.0
Exact duplicate input strings: 1,394


## Export dataset and manifest

In [6]:
if not OVERWRITE and (OUTPUT_PATH.exists() or MANIFEST_PATH.exists()):
    raise FileExistsError('M0 output already exists. Set OVERWRITE=True to replace it intentionally.')

manifest = write_dataset_and_manifest(
    m0, OUTPUT_PATH, MANIFEST_PATH, method=METHOD_M0, input_path=INPUT_PATH,
    parameters={'transformation': 'identity', 'common_source_pool': True},
)
manifest

{'method': 'm0_identity',
 'created_at_utc': '2026-08-03T16:19:16.785371+00:00',
 'input_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\translation data\\train-00000-of-00001.parquet',
 'input_sha256': 'bd4910104fc121ff6924ca9a8b1a0d6bdaa6f28601108703cc9b29b2dc9eb27e',
 'output_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\synthetic\\m0_identity_pairs.csv',
 'output_sha256': '735a8f4e0c27263b5e312a42d550d8a77608c64d7f7a8acdacf2aa1777fb15fb',
 'rows': 17777,
 'unique_sources': 17777,
 'changed_rows': 0,
 'unchanged_rows': 17777,
 'parameters': {'transformation': 'identity', 'common_source_pool': True}}